# 03 — Train local AMR-SAGE (Colab driver)

Phase-1: one hospital → core KG → local GNN, macro-F1 vs the majority baseline.

**Run cells top to bottom** (Runtime → Run all). Only edit the `ARMD_DIR` path in the setup cell.
Per repo convention this notebook only *imports + calls* `src/amr_fed/` modules — no logic here.

In [ ]:
# 1) Get the code (clone the branch, or pull if already cloned)
!git clone -b phase1-core-pipeline https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout phase1-core-pipeline && git pull)
!pip install -q torch_geometric

In [ ]:
# 2) Point at the data. Mount Drive and set ARMD_DIR *before* importing amr_fed.
from google.colab import drive
drive.mount('/content/drive')

import os
# EDIT THIS to your ARMD folder. If ARMD is under 'Shared with me', first add a
# shortcut to it in My Drive, then it appears under /content/drive/MyDrive/.
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'

In [ ]:
# 3) Sanity: config resolves the data dir and sees the CSVs
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix the path in cell 2 and re-run.'
print('CSVs found:', sum((D / f).exists() for f in config.ARMD_TABLES.values()), 'of', len(config.ARMD_TABLES))

In [ ]:
# 4) Smoke test on one small hospital (ICU) — fast end-to-end check
from amr_fed.train_local import main
model, metrics = main(ward='ICU')
print(metrics)

In [ ]:
# 5) Full cohort (all wards) — run once the smoke test prints a macro-F1
model, metrics = main(ward=None)
print(metrics)

In [ ]:
# 6) ENRICHMENT edge #1: add (patient, has, comorbidity). Compare macro-F1 vs core (0.663).
# First run STREAMS the ~18GB comorbidity CSV once (slow — several minutes from Drive)
# and caches the patient->comorbidity edge list to Drive; later runs reuse the cache.
cache = '/content/drive/MyDrive/amr_comorbidity_edges.parquet'
model, metrics = main(ward=None, enrich=('comorbidity',), comorbidity_cache=cache)
print(metrics)